# **Advantages of Cognitive Automation with Generative Models**

Cognitive automation, enhanced by generative models, combines artificial intelligence, machine learning, and advanced data analysis to perform complex tasks with high precision. This approach reduces human errors, standardizes processes, and improves information interpretation, making it especially valuable in sectors such as finance, healthcare, manufacturing, and customer service. In addition to ensuring consistency and reliability in results, it lowers operational costs and rework. With proper training and continuous monitoring, generative models deliver coherent responses, eliminate biases, and strengthen data-driven decision-making.


### Automation of medical processes, specifically for patient triage.

In [ ]:
!pip install --q qdrant-client

In [ ]:
# Imports
import torch
import gradio as gr
import pandas as pd
import transformers
import qdrant_client
import sentence_transformers
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
import warnings
warnings.filterwarnings('ignore')

In [ ]:
#For reproduction of the same results as here
set_seed(1234)

The environment variable below controls tokenization parallelism, i.e., whether multiple threads should be used to process text in parallel. Depending on the environment and workload, enabling or disabling this parallelism can impact performance.

In [ ]:
%env TOKENIZERS_PARALLELISM=True

## Loading Data to the RAG Module

In [ ]:
# Load data into the RAG module
df = pd.read_csv('/kaggle/input/layoutlm/medquad.csv')

In [ ]:
df

In [ ]:
# We'll work with just 10000 records to make the app faster.
# Feel free to work with larger data volumes.
ques_data = df['question'].tolist()[:10000]
answer_data = df['answer'].tolist()[:10000]

## Embeddings Model

https://huggingface.co/sentence-transformers/all-mpnet-base-v2

In [ ]:
# Defines the embeddings model
modelo_embedding = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

## RAG Module with Vector Database

https://qdrant.tech/

In [ ]:
# Use the embeddings model to create vectors from the data
vetores = modelo_embedding.encode(ques_data)

In [ ]:
# Create the client for the vector database defined in memory
banco_vetorial = QdrantClient(":memory:")

In [ ]:
# Create the collection in the vector database
banco_vetorial.create_collection(collection_name = "doc_data",
                                 vectors_config = VectorParams(size = len(vetores[0]),
                                                               distance = Distance.COSINE))

In [ ]:
# Upload data to the vector database
banco_vetorial.upload_collection(collection_name = "doc_data",
                                 ids = [i for i in range(len(ques_data))],
                                 vectors = vetores)

## Information Retrieval Module

In [ ]:
# Define the function that receives a question as input
def dsa_recupera_dados(question):

    # Loads the sentence transformer model "all-mpnet-base-v2" from the SentenceTransformer library
    model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

    # Encodes the question into an embedding vector using the model
    ques_vector = model.encode(question)

    # Queries the vector database with the question vector, searching for similar documents
    result = banco_vetorial.query_points(collection_name="doc_data", query=ques_vector)

    # Creates an empty list to store the IDs of the most similar documents
    sim_ids = []

    # Iterates over the query results and adds the document IDs to the list
    for i in result.points:
        sim_ids.append(i.id)

    # Retrieves the context of the most similar document based on the first ID in the list
    context = answer_data[sim_ids[0]]

    # Returns the document context as the answer
    return context

## Integration Module Between SLM and RAG

https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0

In [ ]:
# Defines the function 'dsa_llm_rag' that receives a question and a context as input
def dsa_llm_rag(question, context):

    # Defines the name of the language model to be used, in this case "TinyLlama-1.1B-Chat-v1.0"
    nome_llm = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

    # Loads the tokenizer associated with the model using the specified name
    tokenizer = AutoTokenizer.from_pretrained(nome_llm)

    # Loads the causal language model using the specified name
    # Replace "cuda" with "cpu" if running locally
    model = AutoModelForCausalLM.from_pretrained(nome_llm, device_map="cuda")

    # Defines the app prompt, which includes the question and the context,
    # instructing the LLM to respond based on the context
    chat = [{"role": "user", "content": f"this is question {question} asked by user you are a medical clinic assistant answer the question based on this context {context} in not more than 3-4 points"}]

    # Applies the chat template, tokenizes the prompt and converts it to PyTorch tensors,
    # adding the generation prompt
    # Remove the final [.to("cuda")] part if running locally
    token_inputs = tokenizer.apply_chat_template(chat,
                                                 tokenize=True,
                                                 return_tensors="pt",
                                                 add_generation_prompt=True).to("cuda")

    # Generates the model's response from the input tokens using sampling
    # and setting limits such as the maximum number of new tokens and temperature (creativity level)
    token_outputs = model.generate(input_ids=token_inputs,
                                   do_sample=True,
                                   max_new_tokens=500,
                                   temperature=1.5)

    # Extracts the new tokens generated that are not part of the original input
    new_tokens = token_outputs[0][token_inputs.shape[-1]:]

    # Decodes the new tokens into text, ignoring special tokens
    decoded_output = tokenizer.decode(new_tokens, skip_special_tokens=True)

    # Returns the decoded text as the answer
    return decoded_output

In [ ]:
# Defines the function 'dsa_gera_resultado' that receives the user input as a parameter
def dsa_gera_resultado(user_input):

    # Checks if the user has provided an input
    if user_input:

        # Retrieves the relevant context by calling the 'dsa_recupera_dados' function with the user input
        context = dsa_recupera_dados(user_input)

        # Generates and returns the answer by calling the 'dsa_llm_rag' function with the user input and retrieved context
        return dsa_llm_rag(user_input, context)

    # If the user input is empty, returns a message asking to enter a question
    else:
        return "Please enter your question."

## Web Application Module To Deploy

In [ ]:
# Defining the custom interface
webapp = gr.Interface(

    # Function that processes the user input
    fn=dsa_gera_resultado,

    # Larger text box with a custom placeholder
    inputs=gr.Textbox(lines=2, placeholder="Type your question here...", label="Input"),

    # Output text box with a custom label
    outputs=gr.Textbox(label="Answer"),

    # Application title
    title="DSA - Project 3",

    # Custom description
    description="This is an AI application for automating the medical patient triage process.",

    # Pre-defined examples
    examples=[["What is (are) Parasites - Schistosomiasis ?"]],

    # More compact theme
    theme="compact"
)

In [ ]:
# Launch the Gradio interface
webapp.launch(share = True)